# Running Large Language Models Locally with Ollama

## Introduction

This notebook is a practical introduction to **running Large Language Models (LLMs) locally** on your own device and using them from **Python scripts** to work with data.

In recent years, LLMs have become popular tools for generating text, answering questions, summarizing documents, extracting information, and assisting with coding. Most people interact with such models through cloud-based services, but it is also possible to run smaller models **locally** on a laptop or desktop computer using tools such as **Ollama**.

Running models locally has several advantages:

- **Privacy:** your data stays on your own machine
- **Accessibility:** you can experiment without relying on external APIs
- **Cost control:** no paid API calls are required for simple demonstrations
- **Transparency:** you better understand how models are integrated into real workflows

The goal of this notebook is to demonstrate two things:

1. **How to run an LLM locally using Ollama**
2. **How to call that model from Python and use it in a small data-processing workflow**

---

## Why this matters for Data Science

In data science, we often work with structured data such as tables, numbers, and clearly defined variables. However, many real-world datasets also contain **unstructured or semi-structured information** such as text, descriptions, notes, labels, or domain-specific strings.

LLMs can be useful in such situations because they can help with tasks like:

- summarizing information
- generating explanations
- extracting structured information from raw text
- classifying content into categories
- assisting with interpretation and annotation

In this notebook, we will use a **chemistry-related dataset** containing molecular representations in the form of **SMILES strings** together with several material-related variables. The purpose is not to build a chemistry prediction model, but rather to show how an LLM can be integrated into a Python workflow to help us **inspect, explain, or process data**.

---

## What is Ollama?

**Ollama** is a tool that makes it easy to download and run LLMs locally. It provides a simple command-line interface and also allows Python scripts to communicate with locally running models.

This means we can:

- start a model on our computer
- send prompts to it
- receive responses programmatically
- integrate the model into data analysis pipelines

This setup is especially useful for teaching and experimentation because it shows that LLMs are not only chat interfaces — they can also be used as components inside scripts and notebooks.

---

## What we will do in this notebook

In this notebook, we will:

- briefly explain the idea of running LLMs locally
- start a small local model with Ollama
- test it with a simple prompt
- connect to the model from Python
- load a chemistry dataset
- use the model in a script to perform a small task on the data

The main purpose is to give an **accessible first example** of how LLMs can be used in practice within a data science workflow.

---

## Important note

Local LLMs can be very useful, but they also have limitations:

- they may produce incorrect or invented information
- smaller local models are usually weaker than large cloud models
- results depend on the prompt and model choice
- LLM outputs should always be checked critically

For that reason, this notebook should be seen as a **demonstration and learning exercise**, not as a scientifically validated chemistry analysis pipeline.

---

## Dataset context

The dataset used here contains:

- an `id` column
- molecular structures encoded as **SMILES strings**
- several numeric columns such as `Tg`, `FFV`, `Tc`, `Density`, and `Rg`

This makes it a good example for exploring how an LLM might assist in handling domain-specific data, even when the data itself is not ordinary natural language.

---

## Learning objective

By the end of this notebook, you should understand:

- what it means to run an LLM locally
- how Ollama can be used for this purpose
- how to send prompts to a local model from Python
- how such a model can be incorporated into a simple data processing workflow

## Setup: Installing the Required Packages

In this notebook, we will use a few Python packages to load data and communicate with a locally running LLM.

We will start with:

- `pandas` for loading and working with tabular data
- `ollama` for sending prompts to a local model from Python

Before running the Python code, make sure that **Ollama is installed on your computer**.

You can download it here:

[https://ollama.com/download](https://ollama.com/download)

Please download and install the version that matches your operating system:

- Windows
- macOS
- Linux

After installing Ollama, open a terminal and make sure it is available on your system.

Later in the notebook, we may install additional packages if needed.

In [ ]:
!pip install pandas ollama

In [ ]:
import pandas as pd
import ollama

print("Packages imported successfully.")

In [ ]:
#install a model. We will choose a very small model that can run on many hardware. Use "qwen3.5:0.8b" if it is too slow
!ollama pull qwen3.5:0.8b

## First interaction with a local LLM

Now that Ollama is installed and a model has been downloaded, we can send our first prompt to the model from Python.

In the code below:

- `model_name` defines which local model we want to use
- `prompt` contains the question or instruction we want to send
- the model generates a response, which we print

You can change the text in the `prompt` variable and try different questions.

In [ ]:
import ollama

model_name = "qwen3.5:2b"
prompt = "Explain in simple words what a large language model is."

response = ollama.chat(
    model=model_name,
    messages=[
        {"role": "user", "content": prompt}
    ]
)

print(response["message"]["content"])

## Using tool calling: let the LLM request a histogram

Now we will connect the local LLM to a small Python tool.

The idea is:

- we define a Python function called `plot_histogram`
- we tell the model that this tool exists
- we ask the model: **"Plot the histogram of the FFV feature"**
- the model decides to call the tool
- Python executes the function and shows the plot

This example is useful because it shows that an LLM can do more than generate text:
it can also act as an interface to code and data analysis tools.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import ollama

# Load the dataset from the same folder as the notebook
df = pd.read_csv("train.csv")

# Choose the local model
model_name = "qwen3.5:2b"

In [ ]:
def plot_histogram(column_name):
    values = pd.to_numeric(df[column_name], errors="coerce").dropna()

    plt.figure(figsize=(7, 4))
    plt.hist(values, bins=30, edgecolor="black")
    plt.title(f"Histogram of {column_name}")
    plt.xlabel(column_name)
    plt.ylabel("Count")
    plt.show()

    return f"Histogram for {column_name} plotted successfully."

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "plot_histogram",
            "description": "Plot a histogram for a numeric column in the dataset.",
            "parameters": {
                "type": "object",
                "properties": {
                    "column_name": {
                        "type": "string",
                        "description": "The name of the numeric column to plot, for example FFV or Tg."
                    }
                },
                "required": ["column_name"]
            }
        }
    }
]

In [ ]:
prompt = "Plot the histogram of the FFV feature."

response = ollama.chat(
    model=model_name,
    messages=[
        {"role": "user", "content": prompt}
    ],
    tools=tools
)

print(response["message"])

In [ ]:
tool_calls = response["message"].get("tool_calls", [])

for tool in tool_calls:
    function_name = tool["function"]["name"]
    arguments = tool["function"]["arguments"]

    if function_name == "plot_histogram":
        result = plot_histogram(arguments["column_name"])
        print(result)

In this example, the model does not directly create the plot by itself.

Instead, it returns a **tool call** telling Python which function to run and with which argument.
Our script then executes that function.

This is an important idea in modern LLM applications:
the model can act as a bridge between natural language instructions and real code.

In [ ]:
#